# Notebook A：軸対称円盤の質量積分と回転曲線［学生用・コード入力版］

**Ryden, *Celestial and Stellar Dynamics*, §6.5**

テキスト付録Aに対応する Notebook です。薄い軸対称円盤を題材に、第1章で学んだ数値積分を別の問題へ適用します。球殻から円環へ質量要素を変え、同じ累積台形則で内包質量を求めます。指数円盤の解析解との比較と格子収束試験によって、実装を再利用した場合も精度を検証します。

後半では、円盤と球対称近似の回転曲線を比較します。ここでは解析式を用い、数値積分の誤差を混ぜずに、質量分布の仮定による違いを調べます。

## 学習目標

- 軸対称円盤の質量を求める問題について、数値積分の被積分関数と積分区間を設定できる。
- 被積分関数を入れ替え、第1章の累積台形則の実装を別の質量分布に再利用できる。
- 解析解との比較と格子収束試験から、累積積分の誤差と収束次数を評価できる。
- 数値積分の離散化誤差と、有限の計算領域の外にある質量を区別できる。
- 数値計算の誤差と、物理モデルの仮定によって生じる違いを区別できる。

第1章の数値積分が前提です。第2・3章や他の付録を先に読む必要はなく、本編の基礎学習や付録Cの軌道計算に進むための必須事項でもありません。積分公式と累積積分は、テキスト第1章と幸谷『Python数値計算プログラミング』第12章を参照してください。

## 学生用：コード入力版の使い方

テキスト『銀河力学で学ぶ数値計算法』v1.2 を開き、**「入力」セルに指定された掲載コードを書き写してから、上から順に実行**します。「入力済み」セルは、準備・描画・結果表示などのコードです。そのまま実行してください。

入力セルにある `raise NotImplementedError(...)` は、未入力で先へ進まないための行です。この1行を削除してコードを入力します。入力前に実行してこのエラーが出るのは正常です。関数や `for`・`if` の内側では、テキストと同じ字下げを保ってください。

コードを実行するには **Shift + Enter** を押します。入力後は保存し、最後にカーネルを再起動して全セルを上から実行すると、入力漏れを確認できます。別の Notebook を先に実行する必要はありません。

ページ番号は v1.2 に対応しています。`p.` は本文に印刷されたページ番号、`PDF` は表紙から数えたページ番号です。同じ計算が短い Listing と完全な関数の両方にある場合は、完全な関数の箇所で一度だけ入力します。

## 入力箇所一覧

| 入力欄 | 内容 | テキスト |
| --- | --- | --- |
| [A-01](#input-A-01) | 指数円盤の面密度と被積分関数 | p.57 |
| [A-02](#input-A-02) | 指数円盤の内包質量 | p.58–59 |
| [A-03](#input-A-03) | 累積台形則の格子収束 | p.60 |
| [A-04](#input-A-04) | 指数円盤の回転曲線 | p.62 |

In [ ]:
# 入力済み：このセルはそのまま実行する。
import numpy as np
import matplotlib.pyplot as plt

G = 1.0

OI = {
    "orange": "#E69F00",
    "sky": "#56B4E9",
    "green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "black": "#000000",
}

plt.rcParams["figure.figsize"] = (6.5, 4.5)
plt.rcParams["font.size"] = 11


# 1. 球対称系と軸対称円盤の質量要素

第1章の球対称系では、球殻の質量要素は
$$
dM = 4\pi r^2\rho(r)\,dr
$$
でした。一方、無限に薄い軸対称円盤では、円盤面内の半径を $R$、単位面積あたりの質量である面密度（surface density）を $\Sigma(R)$ とします。半径 $R$、幅 $dR$ の円環（annulus）の面積は $dA=2\pi R\,dR$ なので、
$$
dM=2\pi R\Sigma(R)\,dR
$$
となります。半径 $R$ より内側の内包質量は
$$
\boxed{M(<R)=\int_0^R 2\pi R'\Sigma(R')\,dR'}
$$
です。$R'$ は積分変数です。

数値積分に渡す被積分関数は $\Sigma(R')$ そのものではなく、$f_M(R')=2\pi R'\Sigma(R')$、積分区間は $[0,R]$ です。球対称系の $4\pi r'^2\rho(r')$ をこの関数に入れ替えれば、第1章と同じ積分法を使えます。

# 2. 指数円盤

Ryden 式(6.160) の指数円盤
$$
\Sigma(R)=\Sigma_0 e^{-R/R_d}
$$
を使います。$\Sigma_0$ は中心面密度、$R_d$ は面密度が中心値の $1/e$ になる半径です。

## 全質量と中心面密度

全質量を $M_d$ とし、$u=R'/R_d$ と変数変換すると、
$$
M_d=2\pi\Sigma_0\int_0^\infty R'e^{-R'/R_d}\,dR'=2\pi\Sigma_0R_d^2\int_0^\infty ue^{-u}\,du
$$
です。部分積分から
$$
\int ue^{-u}\,du=-ue^{-u}+\int e^{-u}\,du=-(u+1)e^{-u}+C
$$
なので、定積分の値は1です。したがって、
$$
M_d=2\pi\Sigma_0R_d^2,\qquad \Sigma_0=\frac{M_d}{2\pi R_d^2}
$$
となります。次のセルでは $M_d$ と $R_d$ を指定して $\Sigma_0$ を決めます。

## 内包質量の解析解

有限半径まで同じ積分を行うと、
$$
M(<R)=2\pi\Sigma_0R_d^2\int_0^{R/R_d}ue^{-u}\,du
$$
から、Ryden 式(6.161) の解析解
$$
\boxed{M(<R)=M_d\left[1-e^{-R/R_d}\left(1+\frac{R}{R_d}\right)\right]}
$$
を得ます。$R=0$ で0、$R\to\infty$ で $M_d$ となります。この解析解を、有限半径までの数値積分を検証する基準にします。

## 被積分関数と実装

被積分関数は
$$
f_M(R)=\frac{dM}{dR}=2\pi R\Sigma(R)=\frac{M_d}{R_d}\frac{R}{R_d}e^{-R/R_d}
$$
です。`sigma_exponential` が面密度、`dMdR_exponential` が被積分関数、`mass_exponential_exact` が内包質量の解析解を返します。`cumulative_trapezoid_manual` は第1章と同じ累積台形則です。

面密度は中心で最大ですが、$dM/dR$ は中心で0です。次の図では、面密度を $\Sigma_0$、被積分関数を $M_d/R_d$ で割り、半径に対する形の違いを確認します。

**入力済み｜円盤のパラメータと補助関数**

In [ ]:
# 入力済み：このセルはそのまま実行する。
Md = 1.0
Rd = 1.0
Sigma0 = Md / (2.0 * np.pi * Rd**2)

def mass_exponential_exact(R, Md=1.0, Rd=1.0):
    x = R / Rd
    return Md * (1.0 - np.exp(-x) * (1.0 + x))

def cumulative_trapezoid_manual(x, y):
    """台形則で y(x) の累積積分を求める（配列操作は Notebook 1 を参照）。"""
    result = np.zeros_like(x, dtype=float)
    dx = np.diff(x)
    area = 0.5 * (y[:-1] + y[1:]) * dx
    result[1:] = np.cumsum(area)
    return result


<a id="input-A-01"></a>
### 入力 A-01｜指数円盤の面密度と被積分関数

**参照：Listing A.1、テキスト v1.2 p.58（PDF 64ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.2.pdf#page=64)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 A-01：指数円盤の面密度と被積分関数
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 A-01 のコードを入力してください。")


In [ ]:
# 入力済み：このセルはそのまま実行する。
R_plot = np.linspace(0.0, 6.0 * Rd, 500)
Sigma_plot = sigma_exponential(R_plot, Md, Rd)
dMdR_plot = dMdR_exponential(R_plot, Md, Rd)

fig, ax = plt.subplots()
ax.plot(R_plot / Rd, Sigma_plot / Sigma0, color=OI["blue"])
ax.set_xlabel(r"$R/R_d$")
ax.set_ylabel(r"$\Sigma(R)/\Sigma_0$")
ax.set_title("Exponential surface-density profile")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots()
ax.plot(R_plot / Rd, dMdR_plot / (Md / Rd), color=OI["vermillion"])
ax.set_xlabel(r"$R/R_d$")
ax.set_ylabel(r"$(dM/dR)/(M_d/R_d)$")
ax.set_title(r"Mass in annuli: $dM/dR = 2\pi R\Sigma(R)$")
ax.grid(alpha=0.25)
plt.show()


# 3. 内包質量の数値積分

第1章と同じ累積台形則を、$f_M(R)=2\pi R\Sigma(R)$ に適用します。計算領域を $0\le R\le R_{\max}$ とし、$N$ 個の区間、$N+1$ 個の格子点に分けます。
$$
R_i=i\Delta R,\qquad \Delta R=\frac{R_{\max}}{N},\qquad i=0,1,\ldots,N.
$$
内包質量の数値近似 $M_i$ は、
$$
M_0=0,\qquad M_{i+1}=M_i+\frac{\Delta R}{2}\left[f_M(R_i)+f_M(R_{i+1})\right]
$$
として外側へ順に求めます。各半径について0から積分をやり直す必要はありません。積分公式と配列演算の対応は、テキスト第1章「累積積分による内包質量プロファイル」を参照してください。

ここでは $R_{\max}=10R_d$、$N=400$ とします。数値解と同じ格子点で解析解を評価し、比較します。

**二つの差を区別する：** 数値解と同じ上端までの解析解との差は積分の誤差です。一方、$M_d-M_{\rm exact}(<R_{\max})$ は計算領域の外にある質量で、有限区間の積分そのものの誤差ではありません。$R_{\max}$ を固定したまま格子を細かくしても、後者は変わりません。以下では両者を $M_d$ で規格化して表示します。

<a id="input-A-02"></a>
### 入力 A-02｜指数円盤の内包質量

**参照：§A.3「内包質量の数値積分」、テキスト v1.2 p.59–60（PDF 65–66ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.2.pdf#page=65)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 A-02：指数円盤の内包質量
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 A-02 のコードを入力してください。")


**入力済み｜内包質量と領域外の質量の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
print(f"M_num(<R_max)   = {M_num[-1]:.10f}")
print(f"M_exact(<R_max) = {M_exact[-1]:.10f}")
print(f"円盤の全質量 = {Md:.10f}")
print(f"外端での符号付き積分誤差 / Md = {mass_error_outer:.6e}")
print(f"計算領域の外にある質量 / Md = {mass_outside:.6e}")


In [ ]:
# 入力済み：このセルはそのまま実行する。
fig, ax = plt.subplots()
ax.plot(R / Rd, M_exact / Md, color=OI["black"], lw=2.0, label="Analytic")
ax.plot(R / Rd, M_num / Md, "--", color=OI["orange"], lw=1.8, label="Numerical")
ax.set_xlabel(r"$R/R_d$")
ax.set_ylabel(r"$M(<R)/M_d$")
ax.set_title("Enclosed mass of an exponential disk")
ax.set_xlim(0, 8)
ax.set_ylim(0, 1.03)
ax.legend()
ax.grid(alpha=0.25)
plt.show()


# 4. 累積台形則の格子収束

計算領域 $R_{\max}=10R_d$ と $M_d,R_d$ を固定し、区間数 $N$ だけを変えます。全格子点での誤差を
$$
\epsilon_M=\max_{0\le i\le N}\frac{|M_{\rm num}(<R_i)-M_{\rm exact}(<R_i)|}{M_d}
$$
で測ります。これは全質量で規格化した最大絶対誤差であり、各点の内包質量に対する相対誤差ではありません。この規格化なら、内包質量が0の中心も評価に含められます。

第1章で学んだ台形則の誤差評価から、滑らかな被積分関数を固定区間で積分すると、離散化誤差が支配する範囲では
$$
\epsilon_M\simeq C\left(\frac{\Delta R}{R_d}\right)^p,\qquad p=2
$$
が期待されます。$R_d$ は固定しているので、コードで使う $\log\Delta R$ に対する $\log\epsilon_M$ の傾きも $p$ です。

$N=25,50,100,200,400,800$ の6格子で計算し、誤差が規則的に減少することを確かめてから傾きを求めます。格子数をさらに増やす場合は、丸め誤差で停滞した点や誤差が0になった点を当てはめに含めないでください。

<a id="input-A-03"></a>
### 入力 A-03｜累積台形則の格子収束

**参照：§A.4「累積台形則の格子収束」、テキスト v1.2 p.61（PDF 67ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.2.pdf#page=67)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 A-03：累積台形則の格子収束
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 A-03 のコードを入力してください。")


**入力済み｜格子幅・誤差・収束次数の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
for N_i, dR_i, err_i in zip(N_values, dR_values, mass_errors):
    print(f"N = {N_i:4d}, dR = {dR_i:.6e}, 誤差 = {err_i:.6e}")

print(f"\n測定した収束次数 = {order_mass:.3f}")


In [ ]:
# 入力済み：このセルはそのまま実行する。
reference = mass_errors[-1] * (dR_values / dR_values[-1])**2

fig, ax = plt.subplots()
ax.loglog(dR_values, mass_errors, "o-", color=OI["blue"], label="Measured error")
ax.loglog(dR_values, reference, "--", color=OI["black"], label=r"$\propto(\Delta R)^2$")
ax.set_xlabel(r"$\Delta R$")
ax.set_ylabel("Maximum normalized mass error")
ax.set_title("Convergence of the enclosed-mass integral")
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()


# 5. 円盤の重力と球対称近似

ここまでは数値積分の誤差を調べました。次に、**数値手法の再利用と、物理の関係式をそのまま使うことは別である**と確認します。

球対称系では Newton の球殻定理により、
$$
v_{\rm circ}^2(r)=\frac{GM(<r)}{r}
$$
です。しかし、円盤面内の重力には $R$ より外側の質量も寄与するため、その半径より内側の総質量だけでは円運動速度を決められません。

Ryden 式(6.145) では、面密度の Hankel 変換を
$$
S(k)=\int_0^\infty\Sigma(R')J_0(kR')R'\,dR'
$$
と定めています。これを使うと、式(6.149) の円運動速度は
$$
v_{\rm circ}^2(R)=2\pi GR\int_0^\infty S(k)J_1(kR)k\,dk
$$
と表されます。$J_0$ と $J_1$ は第1種 Bessel 関数、$k$ は長さの逆数の次元を持つ積分変数です。

この積分表示は、円盤全体の面密度が重力場に関係することを示しています。ここでは Hankel 積分を直接計算せず、次節の指数円盤の解析式を使います。Hankel 積分の直接数値積分は、ここでは扱いません。

# 6. 指数円盤と Mestel 円盤の回転曲線

薄い指数円盤 $\Sigma(R)=\Sigma_0e^{-R/R_d}$ の回転曲線は、修正 Bessel 関数（modified Bessel functions）を用いて
$$
v_{\rm disk}^2(R)=4\pi G\Sigma_0R_d\,y^2\left[I_0(y)K_0(y)-I_1(y)K_1(y)\right],\qquad y=\frac{R}{2R_d}
$$
と書けます。$I_n$ と $K_n$ は第1種・第2種の $n$ 次修正 Bessel 関数です。この式は、Ryden 式(6.149) の Hankel 積分を指数円盤について評価した形です。Ryden 本文にはこの修正 Bessel 関数による式自体は書かれていませんが、図6.6に円盤の回転曲線が示されています。

ここではこの式を導出せず、`scipy.special.iv` と `kv` で値を計算します。比較曲線は
$$
v_{\rm sph}^2(R)=\frac{GM_{\rm exact}(<R)}{R}
$$
とします。内包質量には数値積分値ではなく解析解を使うため、二つの曲線の違いは累積台形則の誤差ではありません。これは質量分布を球対称とみなすことの影響を調べる比較です。

両曲線を $10^{-4}R_d\le R\le6R_d$ の4000点で評価します。`np.argmax` が選ぶ最大位置は格子点の位置であり、補間や最適化で求めた連続的な最大位置ではありません。出力は $v^2$ を $GM_d/R_d$ で規格化します。図では両曲線を共通の値 $v_{\rm max,disk}^2$ で割り、振幅の違いを残します。

## Mestel 円盤と平坦な回転曲線

Ryden 式(6.150) の有限半径で打ち切った Mestel 円盤は、$\Sigma(R)=\Sigma_t R_t/R$（$0<R\le R_t$）です。$M_t=2\pi\Sigma_tR_t^2$ とすると $M(<R)=M_tR/R_t$ なので、$GM(<R)/R$ は一定です。

ただし、円盤の重力場から求めた $v_{\rm circ}^2$ と $GM(<R)/R$ が厳密に一致するのは、無限に広がる Mestel 円盤という特別な場合です。有限半径で打ち切ると、外側の質量の寄与が変わり、端の影響が現れます（Ryden 式6.158–6.159）。

次の図の点線は、**無限 Mestel 円盤の平坦な形を示すための任意規格化の線**です。指数円盤の最大値に合わせただけであり、指数円盤と同じ全質量にした曲線ではありません。有限 Mestel 円盤の重力場は、このコードでは計算していません。

<a id="input-A-04"></a>
### 入力 A-04｜指数円盤の回転曲線

**参照：§A.5「指数円盤と球対称近似の比較」、テキスト v1.2 p.63（PDF 69ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.2.pdf#page=69)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 A-04：指数円盤の回転曲線
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 A-04 のコードを入力してください。")


**入力済み｜回転曲線と最大位置の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
scale_v2 = G * Md / Rd

print(f"指数円盤（厳密解）の最大位置: R/Rd = {Rpeak_disk/Rd:.3f}, v_c^2/(GMd/Rd) = {vc2peak_disk/scale_v2:.4f}")
print(f"球対称近似の最大位置:       R/Rd = {Rpeak_sph/Rd:.3f}, v_c^2/(GMd/Rd) = {vc2peak_sph/scale_v2:.4f}")

# 形状比較のため、無限 Mestel 円盤の平坦な曲線を
# 指数円盤の解析式による曲線の最大値で規格化する。
fig, ax = plt.subplots()
ax.plot(R_rot / Rd, vc2_disk / vc2peak_disk, color=OI["blue"], lw=2.0, label="Exponential disk: exact")
ax.plot(R_rot / Rd, vc2_sph / vc2peak_disk, "--", color=OI["vermillion"], lw=1.8, label="Spherical approximation")
ax.axhline(1.0, color=OI["green"], ls=":", lw=2.0, label="Infinite Mestel disk (flat; arbitrary normalization)")
ax.axvline(Rpeak_disk / Rd, color=OI["blue"], alpha=0.45, ls=":")
ax.axvline(Rpeak_sph / Rd, color=OI["vermillion"], alpha=0.45, ls=":")
ax.set_xlabel(r"$R/R_d$")
ax.set_ylabel(r"$v_{\rm circ}^2/v_{\rm max,disk}^2$")
ax.set_title("Disk geometry and the circular-speed curve")
ax.set_xlim(0, 6)
ax.set_ylim(0, 1.12)
ax.legend()
ax.grid(alpha=0.25)
plt.show()


# まとめ

物理的な題材が変わっても、被積分関数と積分区間を正しく設定すれば、既習の数値積分の実装を再利用できます。ここでは円環の質量要素 $dM=2\pi R\Sigma(R)\,dR$ を使い、解析解との比較と格子収束試験で累積台形則の約2次の収束を確認しました。

数値積分の誤差と、計算領域の外にある質量は別です。また、数値手法を再利用できても、物理の関係式の適用条件まで同じになるわけではありません。回転曲線の比較では解析的な内包質量を用い、球対称近似による違いを数値積分の誤差と切り分けました。Mestel 円盤の比較線は無限円盤の平坦な形を示す任意規格化であり、有限円盤の計算結果ではありません。